# Experiment 3: Neuroadaptive Pressure from Task Variation

Schedules the activation and deactivation of tasks during training, to observe the neuroadaptive changes (can also be done with addition and removal of input/output neurons, but it was felt that this methodology is cleaner due to many choices in how one one treat input/output neuron removal).


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os
import copy
import time
import math
import random
from Dependencies import *

In [ ]:
# Hyperparameters
LEARNING_RATE = 1e-3
BATCH_SIZE = 48
DEVICE = try_gpu(output=True, i=0)
ARCHITECTURE = [28 * 28 * 3, 2500, 2500, 2500, 30]
PSI_DECAY = 1e-1
ADAMW_WEIGHT_DECAY = 1e-3
TOTAL_EPOCHS = 600
REPEAT = 25
NORMALISATION = True
INTRINSIC_LENGTH_APPROACH = "TRAINABLE"
LINEAR_CORRECTION_APPROACH = "TRAINABLE+DECAY"
WEIGHT_INIT = "orthogonal"
SINGULAR_VALUE_THRESHOLD = 0.95 # Just below the orthogonal initialised 1 value
SCAFFOLD_NEURON_THRESHOLD = 5
RANDOM_WIDTH_SEED = 12342397856578
DATA_SPLIT_SEED = 1234
SAVE_DIR = f"./Saved_Models/Experiment 3/A/"

# Desired Scheduling
SCHEDULE_1 = 150
SCHEDULE_2 = 300
SCHEDULE_3 = 450
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Using device: {DEVICE}")
print(f"Architecture template: {ARCHITECTURE}")
print(f"Total epochs: {TOTAL_EPOCHS}")


In [ ]:
def keep_first_n_examples(dataset, n):
    return torch.utils.data.Subset(dataset, np.arange(min(n, len(dataset))))

class FilteredLabelDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, allowed_labels, label_remap):
        self.dataset = dataset
        self.allowed_labels = set(int(label) for label in allowed_labels)
        self.label_remap = {int(key): int(value) for key, value in label_remap.items()}
        self.indices = []

        for idx in range(len(dataset)):
            _, label = dataset[idx]
            label = int(label)
            if label in self.allowed_labels:
                self.indices.append(idx)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        return image, self.label_remap[int(label)]

if NORMALISATION:
    print("Using Normalisation")

    if not os.path.exists("./FMNIST_normalisations.pkl"):
        raw_transform = transforms.Compose([transforms.ToTensor()])
        fmnist_for_stats = datasets.FashionMNIST(root="./data", train=True, download=True, transform=raw_transform)
        fmnist_stack = torch.stack([image for image, label in fmnist_for_stats], dim=0)
        fmnist_mean = fmnist_stack.mean(dim=0).to(torch.float32)
        fmnist_std = fmnist_stack.std(dim=0, unbiased=False).to(torch.float32)
        fmnist_inv_std = torch.where(fmnist_std > 1e-8, 1.0 / fmnist_std, torch.ones_like(fmnist_std))
        pkl.dump({"mean": fmnist_mean, "inverse stddev": fmnist_inv_std}, open("./FMNIST_normalisations.pkl", "wb"))

    if not os.path.exists("./MNIST_normalisations.pkl"):
        raw_transform = transforms.Compose([transforms.ToTensor()])
        mnist_for_stats = datasets.MNIST(root="./data", train=True, download=True, transform=raw_transform)
        mnist_stack = torch.stack([image for image, label in mnist_for_stats], dim=0)
        mnist_mean = mnist_stack.mean(dim=0).to(torch.float32)
        mnist_std = mnist_stack.std(dim=0, unbiased=False).to(torch.float32)
        mnist_inv_std = torch.where(mnist_std > 1e-8, 1.0 / mnist_std, torch.ones_like(mnist_std))
        pkl.dump({"mean": mnist_mean, "inverse stddev": mnist_inv_std}, open("./MNIST_normalisations.pkl", "wb"))

    if not os.path.exists("./EMNIST_AJ_normalisations.pkl"):
        raw_transform = transforms.Compose([transforms.ToTensor()])
        emnist_for_stats_raw = datasets.EMNIST(root="./data", split="letters", train=True, download=True, transform=raw_transform)
        emnist_for_stats = FilteredLabelDataset(
            dataset=emnist_for_stats_raw,
            allowed_labels=range(1, 11),
            label_remap={label: label - 1 for label in range(1, 11)},
        )
        emnist_stack = torch.stack([image for image, label in emnist_for_stats], dim=0)
        emnist_mean = emnist_stack.mean(dim=0).to(torch.float32)
        emnist_std = emnist_stack.std(dim=0, unbiased=False).to(torch.float32)
        emnist_inv_std = torch.where(emnist_std > 1e-8, 1.0 / emnist_std, torch.ones_like(emnist_std))
        pkl.dump({"mean": emnist_mean, "inverse stddev": emnist_inv_std}, open("./EMNIST_AJ_normalisations.pkl", "wb"))

    class MNISTPerPixelNormalize:
        def __init__(self):
            normaliser_dictionary = pkl.load(open("./MNIST_normalisations.pkl", "rb"))
            self.mean = normaliser_dictionary["mean"].to(torch.float32)
            self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

        def __call__(self, tensor):
            return (tensor - self.mean) * self.inv_std

    class FMNISTPerPixelNormalize:
        def __init__(self):
            normaliser_dictionary = pkl.load(open("./FMNIST_normalisations.pkl", "rb"))
            self.mean = normaliser_dictionary["mean"].to(torch.float32)
            self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

        def __call__(self, tensor):
            return (tensor - self.mean) * self.inv_std

    class EMNISTPerPixelNormalize:
        def __init__(self):
            normaliser_dictionary = pkl.load(open("./EMNIST_AJ_normalisations.pkl", "rb"))
            self.mean = normaliser_dictionary["mean"].to(torch.float32)
            self.inv_std = normaliser_dictionary["inverse stddev"].to(torch.float32)

        def __call__(self, tensor):
            return (tensor - self.mean) * self.inv_std

    mnist_transform = transforms.Compose([transforms.ToTensor(), MNISTPerPixelNormalize()])
    fmnist_transform = transforms.Compose([transforms.ToTensor(), FMNISTPerPixelNormalize()])
    emnist_transform = transforms.Compose([transforms.ToTensor(), EMNISTPerPixelNormalize()])
else:
    print("Not using Normalisation")
    mnist_transform = transforms.Compose([transforms.ToTensor()])
    fmnist_transform = transforms.Compose([transforms.ToTensor()])
    emnist_transform = transforms.Compose([transforms.ToTensor()])

mnist_train_full = datasets.MNIST(root="./data", train=True, download=True, transform=mnist_transform)
mnist_test_full = datasets.MNIST(root="./data", train=False, download=True, transform=mnist_transform)

fmnist_train_full = datasets.FashionMNIST(root="./data", train=True, download=True, transform=fmnist_transform)
fmnist_test_full = datasets.FashionMNIST(root="./data", train=False, download=True, transform=fmnist_transform)

emnist_train_raw = datasets.EMNIST(root="./data", split="letters", train=True, download=True, transform=emnist_transform)
emnist_test_raw = datasets.EMNIST(root="./data", split="letters", train=False, download=True, transform=emnist_transform)

emnist_train_filtered = FilteredLabelDataset(
    dataset=emnist_train_raw,
    allowed_labels=range(1, 11),
    label_remap={label: label - 1 for label in range(1, 11)},
)
emnist_test_filtered = FilteredLabelDataset(
    dataset=emnist_test_raw,
    allowed_labels=range(1, 11),
    label_remap={label: label - 1 for label in range(1, 11)},
)

# Keep set sizes consistent: NOTE this is not proportion-aware, which some may find unappealing, but it doesnt hugely affect the neuroadaptive pressure being studied
TARGET_TRAIN_SIZE = min(len(mnist_train_full), len(fmnist_train_full), len(emnist_train_filtered))
TARGET_TEST_SIZE = min(len(mnist_test_full), len(fmnist_test_full), len(emnist_test_filtered))

mnist_train = keep_first_n_examples(mnist_train_full, TARGET_TRAIN_SIZE)
mnist_test = keep_first_n_examples(mnist_test_full, TARGET_TEST_SIZE)

fmnist_train = keep_first_n_examples(fmnist_train_full, TARGET_TRAIN_SIZE)
fmnist_test = keep_first_n_examples(fmnist_test_full, TARGET_TEST_SIZE)

emnist_train = keep_first_n_examples(emnist_train_filtered, TARGET_TRAIN_SIZE)
emnist_test = keep_first_n_examples(emnist_test_filtered, TARGET_TEST_SIZE)


In [ ]:
all_train_sets = {
    "mnist": DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
    "fmnist": DataLoader(fmnist_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
    "emnist": DataLoader(emnist_train, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
}

all_test_sets = {
    "mnist": DataLoader(mnist_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
    "fmnist": DataLoader(fmnist_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
    "emnist": DataLoader(emnist_test, batch_size=BATCH_SIZE, shuffle=False, drop_last=False),
}

DATASET_BLOCKS = {
    "mnist": (0, 28 * 28, 0, 10),
    "fmnist": (28 * 28, 28 * 28 * 2, 10, 20),
    "emnist": (28 * 28 * 2, 28 * 28 * 3, 20, 30),
}

TOTAL_INPUT_WIDTH = 28 * 28 * 3

print("Dataset sizes:")
for dataset_name in ["mnist", "fmnist", "emnist"]:
    print(
        dataset_name,
        len(all_train_sets[dataset_name].dataset),
        len(all_test_sets[dataset_name].dataset),
    )

print(f"Common train size: {TARGET_TRAIN_SIZE}")
print(f"Common test size: {TARGET_TEST_SIZE}")


In [ ]:
def hidden_width_sum(architecture): return int(sum(architecture[1:-1]))

def extract_neuroadaptation_snapshot(network):
    with torch.no_grad():
        architecture = list(network.architecture)
        hidden_sum = hidden_width_sum(architecture)
        singular_values = []
        for layer in range(len(architecture) - 2):
            singular_values.append(network.get_singular_values(layer=layer, use_torch=False).copy())
        if hasattr(network, "psi_parameters"):
            psi_norm_sum = float(sum(torch.linalg.vector_norm(psi.detach(), ord=2).item() for psi in network.psi_parameters))
        else:
            psi_norm_sum = 0.0
    return {
        "architecture": architecture,
        "hidden_width_sum": hidden_sum,
        "singular_values": singular_values,
        "psi_norm_sum": psi_norm_sum,
    }

def active_datasets_for_epoch(epoch):
    if epoch < SCHEDULE_1:
        return ["mnist"]
    elif epoch < SCHEDULE_2:
        return ["mnist", "fmnist"]
    elif epoch < SCHEDULE_3:
        return ["mnist", "fmnist", "emnist"]
    else:
        return ["fmnist"]

def dataset_loss_weights_for_epoch(epoch):
    active_datasets = active_datasets_for_epoch(epoch)
    return {
        "mnist": float("mnist" in active_datasets),
        "fmnist": float("fmnist" in active_datasets),
        "emnist": float("emnist" in active_datasets),
    }

def append_history_dictionary(target_history, value_dictionary):
    for dataset_name in target_history:
        target_history[dataset_name].append(float(value_dictionary[dataset_name]))

def load_ensemble_history(save_dir, repeats):
    histories = []
    for repeat_idx in range(1, repeats + 1):
        with open(os.path.join(save_dir, f"REPEAT_{repeat_idx}.pkl"), "rb") as f:
            histories.append(pkl.load(f))
    return histories


In [ ]:
existing_repeats = set()
for file in os.listdir(SAVE_DIR):
    if file.startswith("REPEAT_") and file.endswith(".pkl"):
        repeat_idx = int(file.split("REPEAT_")[1].split(".pkl")[0])
        existing_repeats.add(repeat_idx)
print(existing_repeats)

In [ ]:
FORCE_RERUN = False
rng = np.random.default_rng(RANDOM_WIDTH_SEED)

# Originally tried a log-based distribution for widths, but uniform became preferable as very small initial widths never had oppurtunity to 'catch-up' before next dataset added.
# repeat_initial_architectures = [
#     [
#         ARCHITECTURE[0],
#         int(np.exp(rng.uniform(1, np.log(ARCHITECTURE[1] + 1)))),
#         int(np.exp(rng.uniform(1, np.log(ARCHITECTURE[2] + 1)))),
#         int(np.exp(rng.uniform(1, np.log(ARCHITECTURE[3] + 1)))),
#         ARCHITECTURE[-1],
#     ]
#     for _ in range(REPEAT)
# ]

repeat_initial_architectures = [
    [
        ARCHITECTURE[0],
        int(rng.uniform(500, ARCHITECTURE[1] + 1)),
        int(rng.uniform(500, ARCHITECTURE[2] + 1)),
        int(rng.uniform(500, ARCHITECTURE[3] + 1)),
        ARCHITECTURE[-1],
    ]
    for _ in range(REPEAT)
]

for repeat in range(REPEAT):
    repeat_idx = repeat + 1
    repeat_architecture = repeat_initial_architectures[repeat]

    if repeat_idx in existing_repeats and not FORCE_RERUN:
        print(f"Skipping Repeat {repeat_idx}/{REPEAT} with initial architecture {repeat_architecture}...")
        continue

    print(f"\n\n========== Repeat {repeat_idx}/{REPEAT} | Initial architecture {repeat_architecture} ==========")

    network = IsotropicTanhMLP(
        layers=repeat_architecture,
        flatten=True,
        unflatten_shape=None,
        intrinsic_length_approach=INTRINSIC_LENGTH_APPROACH,
        linear_correction_approach=LINEAR_CORRECTION_APPROACH,
        positive_intrinsic_length=True,
        init_intrinsic_length=1e-6,
        tanh_epsilon=1e-3,
        device=DEVICE,
        dtype=torch.get_default_dtype(),
    )
    network.simple_initialiser(weight_init=WEIGHT_INIT)
    network.to(DEVICE)

    initial_state_dict = copy.deepcopy(network.state_dict())

    # Static network
    control_network = IsotropicTanhMLP(
        layers=repeat_architecture,
        flatten=True,
        unflatten_shape=None,
        intrinsic_length_approach=INTRINSIC_LENGTH_APPROACH,
        linear_correction_approach=LINEAR_CORRECTION_APPROACH,
        positive_intrinsic_length=True,
        init_intrinsic_length=1e-6,
        tanh_epsilon=1e-3,
        device=DEVICE,
        dtype=torch.get_default_dtype(),
    )
    control_network.load_state_dict(initial_state_dict)
    control_network.to(DEVICE)

    optimiser = torch.optim.AdamW(network.parameters(), lr=LEARNING_RATE, weight_decay=ADAMW_WEIGHT_DECAY)
    control_optimiser = torch.optim.AdamW(control_network.parameters(), lr=LEARNING_RATE, weight_decay=ADAMW_WEIGHT_DECAY)
    loss = nn.CrossEntropyLoss()

    train_x = []
    train_cost = []
    test_x = [0]
    test_cost = []
    architecture_history = []
    hidden_width_sum_history = []
    singular_value_history = []
    psi_train_x = []
    psi_norm_sum_history = []

    control_train_x = []
    control_train_cost = []
    control_test_x = [0]
    control_test_cost = []
    control_architecture_history = []
    control_hidden_width_sum_history = []
    control_singular_value_history = []
    control_psi_train_x = []
    control_psi_norm_sum_history = []

    dataset_test_acc_history = {"mnist": [], "fmnist": [], "emnist": []}
    control_dataset_test_acc_history = {"mnist": [], "fmnist": [], "emnist": []}
    dataset_train_acc_history = {"mnist": [], "fmnist": [], "emnist": []}
    control_dataset_train_acc_history = {"mnist": [], "fmnist": [], "emnist": []}

    dataset_test_loss_history = {"mnist": [], "fmnist": [], "emnist": []}
    control_dataset_test_loss_history = {"mnist": [], "fmnist": [], "emnist": []}
    dataset_train_loss_history = {"mnist": [], "fmnist": [], "emnist": []}
    control_dataset_train_loss_history = {"mnist": [], "fmnist": [], "emnist": []}

    dataset_test_weighted_loss_history = {"mnist": [], "fmnist": [], "emnist": []}
    control_dataset_test_weighted_loss_history = {"mnist": [], "fmnist": [], "emnist": []}
    dataset_train_weighted_loss_history = {"mnist": [], "fmnist": [], "emnist": []}
    control_dataset_train_weighted_loss_history = {"mnist": [], "fmnist": [], "emnist": []}

    initial_loss_weights = dataset_loss_weights_for_epoch(0)

    network, temp_cost, temp_acc_dictionary, temp_loss_dictionary, temp_weighted_loss_dictionary = testing_epoch_variable(
        network=network,
        testing_sets=all_test_sets,
        dataset_blocks=DATASET_BLOCKS,
        total_input_width=network.architecture[0],
        device=DEVICE,
        loss=loss,
        lambda_psi=(PSI_DECAY if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY" else 0.0),
        loss_weights=initial_loss_weights,
    )
    test_cost.append(temp_cost)
    append_history_dictionary(dataset_test_acc_history, temp_acc_dictionary)
    append_history_dictionary(dataset_test_loss_history, temp_loss_dictionary)
    append_history_dictionary(dataset_test_weighted_loss_history, temp_weighted_loss_dictionary)

    control_network, control_temp_cost, control_temp_acc_dictionary, control_temp_loss_dictionary, control_temp_weighted_loss_dictionary = testing_epoch_variable(
        network=control_network,
        testing_sets=all_test_sets,
        dataset_blocks=DATASET_BLOCKS,
        total_input_width=network.architecture[0],
        device=DEVICE,
        loss=loss,
        lambda_psi=(PSI_DECAY if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY" else 0.0),
        loss_weights=initial_loss_weights,
    )
    control_test_cost.append(control_temp_cost)
    append_history_dictionary(control_dataset_test_acc_history, control_temp_acc_dictionary)
    append_history_dictionary(control_dataset_test_loss_history, control_temp_loss_dictionary)
    append_history_dictionary(control_dataset_test_weighted_loss_history, control_temp_weighted_loss_dictionary)

    initial_snapshot = extract_neuroadaptation_snapshot(network)
    architecture_history.append(initial_snapshot["architecture"])
    hidden_width_sum_history.append(initial_snapshot["hidden_width_sum"])
    singular_value_history.append(initial_snapshot["singular_values"])

    control_initial_snapshot = extract_neuroadaptation_snapshot(control_network)
    control_architecture_history.append(control_initial_snapshot["architecture"])
    control_hidden_width_sum_history.append(control_initial_snapshot["hidden_width_sum"])
    control_singular_value_history.append(control_initial_snapshot["singular_values"])

    print(
        f"Initial | Active datasets: {active_datasets_for_epoch(0)} "
        f"| Architecture: {architecture_history[-1]} | Hidden-width sum: {hidden_width_sum_history[-1]}"
    )

    for epoch in range(TOTAL_EPOCHS):
        active_datasets = active_datasets_for_epoch(epoch)
        loss_weights = dataset_loss_weights_for_epoch(epoch)

        print(
            f"Repeat {repeat_idx}/{REPEAT} | Starting Epoch {epoch + 1}/{TOTAL_EPOCHS} | Active datasets: {active_datasets}... ",
            end=""
        )

        adaptive_step_counter = {"i": 0}
        control_step_counter = {"i": 0}

        def adaptive_post_step_hook(*args, **kwargs):
            if hasattr(network, "psi_parameters"):
                psi_norm_value = float(sum(torch.linalg.vector_norm(psi.detach(), ord=2).item() for psi in network.psi_parameters))
            else:
                psi_norm_value = 0.0
            adaptive_total_steps = min(len(loader) for loader in all_train_sets.values())
            psi_train_x.append(epoch + adaptive_step_counter["i"] / adaptive_total_steps)
            psi_norm_sum_history.append(psi_norm_value)
            adaptive_step_counter["i"] += 1

        def control_post_step_hook(*args, **kwargs):
            if hasattr(control_network, "psi_parameters"):
                psi_norm_value = float(sum(torch.linalg.vector_norm(psi.detach(), ord=2).item() for psi in control_network.psi_parameters))
            else:
                psi_norm_value = 0.0
            control_total_steps = min(len(loader) for loader in all_train_sets.values())
            control_psi_train_x.append(epoch + control_step_counter["i"] / control_total_steps)
            control_psi_norm_sum_history.append(psi_norm_value)
            control_step_counter["i"] += 1

        adaptive_hook_handle = optimiser.register_step_post_hook(adaptive_post_step_hook)
        control_hook_handle = control_optimiser.register_step_post_hook(control_post_step_hook)

        network, epoch_x, epoch_cost, epoch_acc_dictionary, epoch_loss_dictionary, epoch_weighted_loss_dictionary = training_epoch_variable(
            network=network,
            training_sets=all_train_sets,
            dataset_blocks=DATASET_BLOCKS,
            total_input_width=network.architecture[0],
            device=DEVICE,
            optimiser=optimiser,
            loss=loss,
            current_epoch=epoch,
            lambda_psi=(PSI_DECAY if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY" else 0.0),
            loss_weights=loss_weights,
        )

        control_network, control_epoch_x, control_epoch_cost, control_epoch_acc_dictionary, control_epoch_loss_dictionary, control_epoch_weighted_loss_dictionary = training_epoch_variable(
            network=control_network,
            training_sets=all_train_sets,
            dataset_blocks=DATASET_BLOCKS,
            total_input_width=control_network.architecture[0],
            device=DEVICE,
            optimiser=control_optimiser,
            loss=loss,
            current_epoch=epoch,
            lambda_psi=(PSI_DECAY if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY" else 0.0),
            loss_weights=loss_weights,
        )

        adaptive_hook_handle.remove()
        control_hook_handle.remove()

        train_x += epoch_x
        train_cost += epoch_cost
        control_train_x += control_epoch_x
        control_train_cost += control_epoch_cost

        for dataset_name in dataset_train_acc_history:
            dataset_train_acc_history[dataset_name].append(float(np.mean(epoch_acc_dictionary[dataset_name])))
            control_dataset_train_acc_history[dataset_name].append(float(np.mean(control_epoch_acc_dictionary[dataset_name])))

            dataset_train_loss_history[dataset_name].append(float(np.mean(epoch_loss_dictionary[dataset_name])))
            control_dataset_train_loss_history[dataset_name].append(float(np.mean(control_epoch_loss_dictionary[dataset_name])))

            dataset_train_weighted_loss_history[dataset_name].append(float(np.mean(epoch_weighted_loss_dictionary[dataset_name])))
            control_dataset_train_weighted_loss_history[dataset_name].append(float(np.mean(control_epoch_weighted_loss_dictionary[dataset_name])))

        for layer in range(len(network.architecture) - 3, -1, -1):
            network.auto_neuroadapation(
                layer=layer,
                scaffold_neuron_threshold=SCAFFOLD_NEURON_THRESHOLD,
                singular_value_threshold=float(SINGULAR_VALUE_THRESHOLD),
                verbose=False,
                optimiser=optimiser,
            )

        network, temp_cost, temp_acc_dictionary, temp_loss_dictionary, temp_weighted_loss_dictionary = testing_epoch_variable(
            network=network,
            testing_sets=all_test_sets,
            dataset_blocks=DATASET_BLOCKS,
            total_input_width=network.architecture[0],
            device=DEVICE,
            loss=loss,
            lambda_psi=(PSI_DECAY if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY" else 0.0),
            loss_weights=loss_weights,
        )
        test_x.append(epoch + 1)
        test_cost.append(temp_cost)
        append_history_dictionary(dataset_test_acc_history, temp_acc_dictionary)
        append_history_dictionary(dataset_test_loss_history, temp_loss_dictionary)
        append_history_dictionary(dataset_test_weighted_loss_history, temp_weighted_loss_dictionary)

        control_network, control_temp_cost, control_temp_acc_dictionary, control_temp_loss_dictionary, control_temp_weighted_loss_dictionary = testing_epoch_variable(
            network=control_network,
            testing_sets=all_test_sets,
            dataset_blocks=DATASET_BLOCKS,
            total_input_width=control_network.architecture[0],
            device=DEVICE,
            loss=loss,
            lambda_psi=(PSI_DECAY if LINEAR_CORRECTION_APPROACH == "TRAINABLE+DECAY" else 0.0),
            loss_weights=loss_weights,
        )
        control_test_x.append(epoch + 1)
        control_test_cost.append(control_temp_cost)
        append_history_dictionary(control_dataset_test_acc_history, control_temp_acc_dictionary)
        append_history_dictionary(control_dataset_test_loss_history, control_temp_loss_dictionary)
        append_history_dictionary(control_dataset_test_weighted_loss_history, control_temp_weighted_loss_dictionary)

        epoch_snapshot = extract_neuroadaptation_snapshot(network)
        architecture_history.append(epoch_snapshot["architecture"])
        hidden_width_sum_history.append(epoch_snapshot["hidden_width_sum"])
        singular_value_history.append(epoch_snapshot["singular_values"])

        control_epoch_snapshot = extract_neuroadaptation_snapshot(control_network)
        control_architecture_history.append(control_epoch_snapshot["architecture"])
        control_hidden_width_sum_history.append(control_epoch_snapshot["hidden_width_sum"])
        control_singular_value_history.append(control_epoch_snapshot["singular_values"])

        adaptive_last_psi = psi_norm_sum_history[-1] if len(psi_norm_sum_history) > 0 else 0.0
        control_last_psi = control_psi_norm_sum_history[-1] if len(control_psi_norm_sum_history) > 0 else 0.0

        print(
            f"A/C test cost: {test_cost[-1]:5.4f}/{control_test_cost[-1]:5.4f} "
            f"| A/C width sum: {hidden_width_sum_history[-1]}/{control_hidden_width_sum_history[-1]} "
            f"| A/C Psi L2 sum: {adaptive_last_psi:5.4f}/{control_last_psi:5.4f} | A/C Architecture {network.architecture}/{control_network.architecture}"
        )

        current_dataset_accuracy_summary = " | ".join(
            f"{dataset_name.upper()} acc A/C: {temp_acc_dictionary[dataset_name]:5.2f}%/{control_temp_acc_dictionary[dataset_name]:5.2f}%"
            for dataset_name in dataset_test_acc_history
        )
        print(current_dataset_accuracy_summary)

    repeat_history = {
        "repeat_idx": repeat_idx,
        "hyperparameters": {
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "total_epochs": TOTAL_EPOCHS,
            "initial_architecture": repeat_architecture,
            "initial_architecture_template": ARCHITECTURE,
            "psi_decay": PSI_DECAY,
            "intrinsic_length_approach": INTRINSIC_LENGTH_APPROACH,
            "linear_correction_approach": LINEAR_CORRECTION_APPROACH,
            "weight_init": WEIGHT_INIT,
            "singular_value_threshold": SINGULAR_VALUE_THRESHOLD,
            "scaffold_neuron_threshold": SCAFFOLD_NEURON_THRESHOLD,
        },
        "network": network,
        "control_network": control_network,
        "final_network_state_dict": {k: v.detach().cpu().clone() for k, v in network.state_dict().items()},
        "final_control_network_state_dict": {k: v.detach().cpu().clone() for k, v in control_network.state_dict().items()},
        "initial_state_dict": {k: v.detach().cpu().clone() for k, v in initial_state_dict.items()},
        "train_x": train_x,
        "train_cost": train_cost,
        "test_x": test_x,
        "test_cost": test_cost,
        "architectures": architecture_history,
        "hidden_width_sum": hidden_width_sum_history,
        "singular_values": singular_value_history,
        "psi_train_x": psi_train_x,
        "psi_norm_sum": psi_norm_sum_history,
        "control_train_x": control_train_x,
        "control_train_cost": control_train_cost,
        "control_test_x": control_test_x,
        "control_test_cost": control_test_cost,
        "control_architectures": control_architecture_history,
        "control_hidden_width_sum": control_hidden_width_sum_history,
        "control_singular_values": control_singular_value_history,
        "control_psi_train_x": control_psi_train_x,
        "control_psi_norm_sum": control_psi_norm_sum_history,
        "train_acc_mnist": dataset_train_acc_history["mnist"],
        "train_acc_fmnist": dataset_train_acc_history["fmnist"],
        "train_acc_emnist": dataset_train_acc_history["emnist"],
        "test_acc_mnist": dataset_test_acc_history["mnist"],
        "test_acc_fmnist": dataset_test_acc_history["fmnist"],
        "test_acc_emnist": dataset_test_acc_history["emnist"],
        "train_loss_mnist": dataset_train_loss_history["mnist"],
        "train_loss_fmnist": dataset_train_loss_history["fmnist"],
        "train_loss_emnist": dataset_train_loss_history["emnist"],
        "test_loss_mnist": dataset_test_loss_history["mnist"],
        "test_loss_fmnist": dataset_test_loss_history["fmnist"],
        "test_loss_emnist": dataset_test_loss_history["emnist"],
        "train_weighted_loss_mnist": dataset_train_weighted_loss_history["mnist"],
        "train_weighted_loss_fmnist": dataset_train_weighted_loss_history["fmnist"],
        "train_weighted_loss_emnist": dataset_train_weighted_loss_history["emnist"],
        "test_weighted_loss_mnist": dataset_test_weighted_loss_history["mnist"],
        "test_weighted_loss_fmnist": dataset_test_weighted_loss_history["fmnist"],
        "test_weighted_loss_emnist": dataset_test_weighted_loss_history["emnist"],
        "control_train_acc_mnist": control_dataset_train_acc_history["mnist"],
        "control_train_acc_fmnist": control_dataset_train_acc_history["fmnist"],
        "control_train_acc_emnist": control_dataset_train_acc_history["emnist"],
        "control_test_acc_mnist": control_dataset_test_acc_history["mnist"],
        "control_test_acc_fmnist": control_dataset_test_acc_history["fmnist"],
        "control_test_acc_emnist": control_dataset_test_acc_history["emnist"],
        "control_train_loss_mnist": control_dataset_train_loss_history["mnist"],
        "control_train_loss_fmnist": control_dataset_train_loss_history["fmnist"],
        "control_train_loss_emnist": control_dataset_train_loss_history["emnist"],
        "control_test_loss_mnist": control_dataset_test_loss_history["mnist"],
        "control_test_loss_fmnist": control_dataset_test_loss_history["fmnist"],
        "control_test_loss_emnist": control_dataset_test_loss_history["emnist"],
        "control_train_weighted_loss_mnist": control_dataset_train_weighted_loss_history["mnist"],
        "control_train_weighted_loss_fmnist": control_dataset_train_weighted_loss_history["fmnist"],
        "control_train_weighted_loss_emnist": control_dataset_train_weighted_loss_history["emnist"],
        "control_test_weighted_loss_mnist": control_dataset_test_weighted_loss_history["mnist"],
        "control_test_weighted_loss_fmnist": control_dataset_test_weighted_loss_history["fmnist"],
        "control_test_weighted_loss_emnist": control_dataset_test_weighted_loss_history["emnist"],
    }

    with open(os.path.join(SAVE_DIR, f"REPEAT_{repeat_idx}.pkl"), "wb") as f:
        pkl.dump(repeat_history, f)


Plots:

In [ ]:
histories = load_ensemble_history(SAVE_DIR, REPEAT)

In [ ]:
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ----------------------------
# Styling
# ----------------------------
mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "grid.linewidth": 0.5,
    "grid.alpha": 0.22,
    "lines.linewidth": 1.8,
    "savefig.dpi": 300,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.labelcolor": "black",
    "axes.edgecolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
})

# ----------------------------
# Inputs already defined in notebook:
# histories
# optionally SCHEDULE_1, SCHEDULE_2, SCHEDULE_3
# ----------------------------
S1 = SCHEDULE_1 if "SCHEDULE_1" in globals() else 150
S2 = SCHEDULE_2 if "SCHEDULE_2" in globals() else 300
S3 = SCHEDULE_3 if "SCHEDULE_3" in globals() else 450

sigma_mult = 1.0

os.makedirs("./TempImages", exist_ok=True)
out_svg = "./TempImages/DifferentDatasets.svg"

# ----------------------------
# Utilities
# ----------------------------
def pick_xticks(start_epoch, end_epoch):
    span = end_epoch - start_epoch + 1
    if span <= 20:
        step = 2
    elif span <= 60:
        step = 5
    elif span <= 150:
        step = 10
    elif span <= 300:
        step = 25
    else:
        step = 50
    ticks = list(range(start_epoch, end_epoch + 1, step))
    if ticks[-1] != end_epoch:
        ticks.append(end_epoch)
    return ticks

def mean_std_count(arr_2d):
    means = arr_2d.mean(axis=0)
    if arr_2d.shape[0] > 1:
        stds = arr_2d.std(axis=0, ddof=1)
    else:
        stds = np.zeros(arr_2d.shape[1], dtype=np.float64)
    counts = np.full(arr_2d.shape[1], arr_2d.shape[0], dtype=int)
    return means, stds, counts

def compute_ylim(series_list, sigma_mult=1.0, symmetric=False, pad_fraction=0.06, min_pad=0.6):
    vals = []
    for means, stds, counts in series_list:
        for m, s, n in zip(means, stds, counts):
            if n > 1:
                vals.extend([m - sigma_mult * s, m + sigma_mult * s])
            else:
                vals.append(m)

    if not vals:
        return None

    if symmetric:
        lim = max(abs(v) for v in vals)
        lim = max(lim, 0.75)
        lim *= 1.04
        return (-lim, lim)

    y_min = min(vals)
    y_max = max(vals)
    span = max(y_max - y_min, 1e-8)
    pad = max(min_pad, span * pad_fraction)
    return (y_min - pad, y_max + pad)

def compute_ylim_from_arrays(array_list, symmetric=False, pad_fraction=0.06, min_pad=0.6):
    vals = []
    for arr in array_list:
        arr = np.asarray(arr)
        if arr.size > 0:
            vals.append(arr[np.isfinite(arr)].ravel())

    if not vals:
        return None

    vals = np.concatenate(vals)

    if symmetric:
        lim = max(abs(v) for v in vals)
        lim = max(lim, 0.75)
        lim *= 1.04
        return (-lim, lim)

    y_min = np.min(vals)
    y_max = np.max(vals)
    span = max(y_max - y_min, 1e-8)
    pad = max(min_pad, span * pad_fraction)
    return (y_min - pad, y_max + pad)

def plot_mean_sigma(ax, x, means, stds, counts, label, color, sigma_mult=1.0,
                    linestyle="-", band_alpha=0.12):
    ax.plot(
        x, means,
        color=color,
        linestyle=linestyle,
        linewidth=1.8 if linestyle == "-" else 1.5,
        label=label,
        zorder=3,
    )

    upper = means + sigma_mult * stds
    lower = means - sigma_mult * stds

    if np.any(counts > 1):
        mask = counts > 1
        start = None
        for i in range(len(mask) + 1):
            if i < len(mask) and mask[i] and start is None:
                start = i
            if (i == len(mask) or not mask[i]) and start is not None:
                sl = slice(start, i)
                ax.plot(x[sl], upper[sl], linestyle=":", linewidth=1.0,
                        color=color, alpha=0.9, zorder=2)
                ax.plot(x[sl], lower[sl], linestyle=":", linewidth=1.0,
                        color=color, alpha=0.9, zorder=2)
                ax.fill_between(x[sl], lower[sl], upper[sl], color=color,
                                alpha=band_alpha, linewidth=0, zorder=1)
                start = None

def plot_individual_runs(ax, x, arr_2d, label, color, alpha=0.5, linewidth=1.15):
    for i in range(arr_2d.shape[0]):
        ax.plot(
            x,
            arr_2d[i],
            color=color,
            alpha=alpha,
            linewidth=linewidth,
            label=label if i == 0 else None,
            zorder=2
        )

def add_schedule_markers(ax, short_labels=True):
    markers = [
        (S1, "T1" if short_labels else "MNIST\nFMNIST"),
        (S2, "T2" if short_labels else "MNIST\nFMNIST\nEMNIST"),
        (S3, "T3" if short_labels else "FMNIST"),
    ]

    y0, y1 = ax.get_ylim()
    y_text = y1 - 0.03 * (y1 - y0)

    for x, txt in markers:
        ax.axvline(x, color="0.50", linestyle=(0, (4, 3)), linewidth=0.9, alpha=0.85, zorder=0)
        ax.text(
            x + 2,
            y_text,
            txt,
            ha="left",
            va="top",
            fontsize=7,
            color="0.30",
            bbox=dict(
                boxstyle="round,pad=0.12",
                facecolor="white",
                edgecolor="none",
                alpha=0.80
            ),
            zorder=5,
        )

def style_axis(ax, xlim, xticks):
    ax.set_xlim(*xlim)
    ax.set_xticks(xticks)
    ax.grid(True, which="major", axis="both")
    ax.tick_params(direction="out", length=3, width=0.8, colors="black")
    ax.margins(x=0)

# ----------------------------
# Read directly from histories
# ----------------------------
adaptive_arch = np.asarray([h["architectures"] for h in histories], dtype=np.float64)
adaptive_width_sum = np.asarray([h["hidden_width_sum"] for h in histories], dtype=np.float64)

mnist_adapt  = np.asarray([h["test_acc_mnist"] for h in histories], dtype=np.float64)
mnist_ctrl   = np.asarray([h["control_test_acc_mnist"] for h in histories], dtype=np.float64)
fmnist_adapt = np.asarray([h["test_acc_fmnist"] for h in histories], dtype=np.float64)
fmnist_ctrl  = np.asarray([h["control_test_acc_fmnist"] for h in histories], dtype=np.float64)
emnist_adapt = np.asarray([h["test_acc_emnist"] for h in histories], dtype=np.float64)
emnist_ctrl  = np.asarray([h["control_test_acc_emnist"] for h in histories], dtype=np.float64)

# adaptive widths
h1 = adaptive_arch[:, :, 1]
h2 = adaptive_arch[:, :, 2]
h3 = adaptive_arch[:, :, 3]
tot = adaptive_width_sum

# epoch axes
epochs_width = np.arange(tot.shape[1])
epochs_acc = np.arange(mnist_adapt.shape[1])

max_epoch = int(max(epochs_width[-1], epochs_acc[-1]))
xticks = pick_xticks(0, max_epoch)

# aggregates
h1_m, h1_s, h1_n = mean_std_count(h1)
h2_m, h2_s, h2_n = mean_std_count(h2)
h3_m, h3_s, h3_n = mean_std_count(h3)
tot_m, tot_s, tot_n = mean_std_count(tot)

mnist_adapt_m, mnist_adapt_s, mnist_adapt_n = mean_std_count(mnist_adapt)
mnist_ctrl_m,  mnist_ctrl_s,  mnist_ctrl_n  = mean_std_count(mnist_ctrl)
fmnist_adapt_m, fmnist_adapt_s, fmnist_adapt_n = mean_std_count(fmnist_adapt)
fmnist_ctrl_m,  fmnist_ctrl_s,  fmnist_ctrl_n  = mean_std_count(fmnist_ctrl)
emnist_adapt_m, emnist_adapt_s, emnist_adapt_n = mean_std_count(emnist_adapt)
emnist_ctrl_m,  emnist_ctrl_s,  emnist_ctrl_n  = mean_std_count(emnist_ctrl)

# colours
colors_width = {
    "tot": "black",
    "h1": "#1f77b4",
    "h2": "#ff7f0e",
    "h3": "#2ca02c",
}

# Distinct from width colours
colors_acc = {
    "mnist":  "#9467bd",  # purple
    "fmnist": "#8c564b",  # brown
    "emnist": "#e377c2",  # pink
}

# ----------------------------
# Plot: left, centre, right
# ----------------------------
fig, axes = plt.subplots(
    1, 3,
    figsize=(16.5, 3.5),
    constrained_layout=True,
    gridspec_kw={"width_ratios": [1.0, 1.0, 1.15]}
)
ax0, ax1, ax2 = axes

# (a) Individual run layer widths
plot_individual_runs(ax0, epochs_width, h1,  "H1",    colors_width["h1"], alpha=0.25, linewidth=1.05)
plot_individual_runs(ax0, epochs_width, h2,  "H2",    colors_width["h2"], alpha=0.25, linewidth=1.05)
plot_individual_runs(ax0, epochs_width, h3,  "H3",    colors_width["h3"], alpha=0.25, linewidth=1.05)
plot_individual_runs(ax0, epochs_width, tot, "Total", colors_width["tot"], alpha=0.25, linewidth=1.15)

ax0.set_title("(a) Individual Run - Layer Widths")
ax0.set_xlabel("Epoch")
ax0.set_ylabel("Units")
style_axis(ax0, (0, max_epoch), xticks)
ax0.set_ylim(*compute_ylim_from_arrays(
    [tot, h1, h2, h3],
    symmetric=False,
    pad_fraction=0.035,
    min_pad=8.0,
))
add_schedule_markers(ax0, short_labels=True)

# (b) Ensemble layer widths
plot_mean_sigma(ax1, epochs_width, tot_m, tot_s, tot_n, "Total", colors_width["tot"],
                sigma_mult=sigma_mult, band_alpha=0.14)
plot_mean_sigma(ax1, epochs_width, h1_m, h1_s, h1_n, "H1", colors_width["h1"],
                sigma_mult=sigma_mult, band_alpha=0.14)
plot_mean_sigma(ax1, epochs_width, h2_m, h2_s, h2_n, "H2", colors_width["h2"],
                sigma_mult=sigma_mult, band_alpha=0.14)
plot_mean_sigma(ax1, epochs_width, h3_m, h3_s, h3_n, "H3", colors_width["h3"],
                sigma_mult=sigma_mult, band_alpha=0.14)

ax1.set_title("(b) Ensemble - Layer Widths")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Units")
style_axis(ax1, (0, max_epoch), xticks)
ax1.set_ylim(*compute_ylim(
    [(tot_m, tot_s, tot_n), (h1_m, h1_s, h1_n), (h2_m, h2_s, h2_n), (h3_m, h3_s, h3_n)],
    sigma_mult=sigma_mult,
    symmetric=False,
    pad_fraction=0.035,
    min_pad=8.0,
))
add_schedule_markers(ax1, short_labels=True)

# (c) Test accuracy
plot_mean_sigma(ax2, epochs_acc, mnist_adapt_m, mnist_adapt_s, mnist_adapt_n,
                "MNIST adaptive", colors_acc["mnist"], sigma_mult=sigma_mult,
                linestyle="-", band_alpha=0.10)
plot_mean_sigma(ax2, epochs_acc, mnist_ctrl_m,  mnist_ctrl_s,  mnist_ctrl_n,
                "MNIST control", colors_acc["mnist"], sigma_mult=sigma_mult,
                linestyle="--", band_alpha=0.08)

plot_mean_sigma(ax2, epochs_acc, fmnist_adapt_m, fmnist_adapt_s, fmnist_adapt_n,
                "FMNIST adaptive", colors_acc["fmnist"], sigma_mult=sigma_mult,
                linestyle="-", band_alpha=0.10)
plot_mean_sigma(ax2, epochs_acc, fmnist_ctrl_m,  fmnist_ctrl_s,  fmnist_ctrl_n,
                "FMNIST control", colors_acc["fmnist"], sigma_mult=sigma_mult,
                linestyle="--", band_alpha=0.08)

plot_mean_sigma(ax2, epochs_acc, emnist_adapt_m, emnist_adapt_s, emnist_adapt_n,
                "EMNIST adaptive", colors_acc["emnist"], sigma_mult=sigma_mult,
                linestyle="-", band_alpha=0.10)
plot_mean_sigma(ax2, epochs_acc, emnist_ctrl_m,  emnist_ctrl_s,  emnist_ctrl_n,
                "EMNIST control", colors_acc["emnist"], sigma_mult=sigma_mult,
                linestyle="--", band_alpha=0.08)

ax2.set_title("(c) Test Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy (%)")
style_axis(ax2, (0, max_epoch), xticks)
ax2.set_ylim(*compute_ylim(
    [
        (mnist_adapt_m, mnist_adapt_s, mnist_adapt_n),
        (mnist_ctrl_m,  mnist_ctrl_s,  mnist_ctrl_n),
        (fmnist_adapt_m, fmnist_adapt_s, fmnist_adapt_n),
        (fmnist_ctrl_m,  fmnist_ctrl_s,  fmnist_ctrl_n),
        (emnist_adapt_m, emnist_adapt_s, emnist_adapt_n),
        (emnist_ctrl_m,  emnist_ctrl_s,  emnist_ctrl_n),
    ],
    sigma_mult=sigma_mult,
    symmetric=False,
    pad_fraction=0.045,
    min_pad=1.0,
))
add_schedule_markers(ax2, short_labels=True)

# ----------------------------
# Legends
# ----------------------------
width_handles = [
    Line2D([0], [0], color=colors_width["tot"], lw=1.8, label="Total"),
    Line2D([0], [0], color=colors_width["h1"], lw=1.8, label="H1"),
    Line2D([0], [0], color=colors_width["h2"], lw=1.8, label="H2"),
    Line2D([0], [0], color=colors_width["h3"], lw=1.8, label="H3"),
]
leg_ax1 = ax1.legend(
    handles=width_handles,
    ncol=2,
    frameon=False,
    loc="lower right",
    handlelength=2.2,
    columnspacing=1.0,
    handletextpad=0.5,
)
for t in leg_ax1.get_texts():
    t.set_color("black")

dataset_handles = [
    Line2D([0], [0], color=colors_acc["mnist"], lw=1.8, label="MNIST"),
    Line2D([0], [0], color=colors_acc["fmnist"], lw=1.8, label="FMNIST"),
    Line2D([0], [0], color=colors_acc["emnist"], lw=1.8, label="EMNIST"),
]
style_handles = [
    Line2D([0], [0], color="0.2", lw=1.8, linestyle="-", label="Adaptive"),
    Line2D([0], [0], color="0.2", lw=1.5, linestyle="--", label="Control"),
    Line2D([0], [0], color="0.2", lw=1.0, linestyle=":", label=rf"$\pm {sigma_mult:g}\sigma$"),
]

leg1 = ax2.legend(
    handles=dataset_handles,
    ncol=1,
    frameon=False,
    loc="lower left",
    bbox_to_anchor=(0.01, 0.01),
    handlelength=2.2,
    handletextpad=0.5,
)
for t in leg1.get_texts():
    t.set_color("black")
ax2.add_artist(leg1)

leg2 = ax2.legend(
    handles=style_handles,
    ncol=1,
    frameon=False,
    loc="lower right",
    bbox_to_anchor=(0.99, 0.01),
    handlelength=2.2,
    handletextpad=0.5,
)
for t in leg2.get_texts():
    t.set_color("black")

# ----------------------------
# Save / show
# ----------------------------
fig.savefig(out_svg, bbox_inches="tight", facecolor="white")
plt.show()

print({
    "out_svg": out_svg,
    "n_repeats": len(histories),
    "schedule_epochs": [S1, S2, S3],
    "max_epoch": max_epoch,
})

In [ ]:
import os
import pickle as pkl
import numpy as np
import pandas as pd

SAVE_DIR = "./Saved_Models/Experiment 3/A/"
REPEAT = 25

histories = []
for repeat_idx in range(1, REPEAT + 1):
    with open(os.path.join(SAVE_DIR, f"REPEAT_{repeat_idx}.pkl"), "rb") as f:
        histories.append(pkl.load(f))

# Loop epochs just before transition: 149, 299, 449, 599.
# Logged after those epochs at test_x/index: 150, 300, 450, 600.
extract_points = {
    "after_loop_epoch_149 / test_x_150": 150,
    "after_loop_epoch_299 / test_x_300": 300,
    "after_loop_epoch_449 / test_x_450": 450,
    "after_loop_epoch_599 / test_x_600": 600,
}

datasets = ["mnist", "fmnist", "emnist"]

rows = []

for label, idx in extract_points.items():
    for dataset in datasets:
        adaptive_key = f"test_acc_{dataset}"
        control_key = f"control_test_acc_{dataset}"

        adaptive_values = np.asarray(
            [h[adaptive_key][idx] for h in histories],
            dtype=np.float64,
        )
        control_values = np.asarray(
            [h[control_key][idx] for h in histories],
            dtype=np.float64,
        )

        rows.append({
            "timepoint": label,
            "index": idx,
            "dataset": dataset.upper(),
            "adaptive_mean": adaptive_values.mean(),
            "adaptive_std": adaptive_values.std(ddof=1),
            "control_mean": control_values.mean(),
            "control_std": control_values.std(ddof=1),
            "n": len(histories),
        })

summary = pd.DataFrame(rows)

summary["adaptive_mean±std"] = summary.apply(
    lambda r: f"{r['adaptive_mean']:.1f} ± {r['adaptive_std']/math.sqrt(len(histories)):.1f}",
    axis=1,
)
summary["control_mean±std"] = summary.apply(
    lambda r: f"{r['control_mean']:.1f} ± {r['control_std']/math.sqrt(len(histories)):.1f}",
    axis=1,
)

summary[
    [
        "timepoint",
        "dataset",
        "adaptive_mean±std",
        "control_mean±std",
        "n",
    ]
]